# Notebook 01 — Data Ingestion and Preprocessing

**Purpose:** Load all raw CSVs, validate them, and save a single clean pickle
(`optimization_inputs.pkl`) that every downstream notebook reads from.



---
**Outputs:**
- `data/processed/optimization_inputs.pkl` — dict with keys:
  `farmers_15`, `farmers_25`, `practices`, `Alpha`, `Beta`, `Gamma`, `Delta`,
  `practice_names`, `practice_index`, `config`

## 0. Imports and setup

In [1]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

# Make sure config.py is importable regardless of working directory
sys.path.insert(0, str(Path.cwd()))
import config

# Create processed output directory if it doesn't exist
Path(config.PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

print("Python path OK")
print(f"Data directory   : {config.DATA_DIR}")
print(f"Processed dir    : {config.PROCESSED_DIR}")

Python path OK
Data directory   : /Users/rohitsuresh/Desktop/CGT_Carbon_farming/data/Dataset_INR
Processed dir    : /Users/rohitsuresh/Desktop/CGT_Carbon_farming/data/processed/Experiment_01_Baseline


## 1. Load farmer data

In [2]:
farmers_15 = pd.read_csv(config.FARMERS_15_CSV)
farmers_25 = pd.read_csv(config.FARMERS_25_CSV)

print(f"15-farmer dataset shape : {farmers_15.shape}")
print(f"25-farmer dataset shape : {farmers_25.shape}")
print()
print("Columns:", list(farmers_15.columns))
print()
farmers_15.head()

15-farmer dataset shape : (15, 4)
25-farmer dataset shape : (25, 4)

Columns: ['Farmer_ID', 'Farm_Size_ha', 'Budget_per_ha_INR_per_season', 'Budget_total_INR_per_season']



,Farmer_ID,Farm_Size_ha,Budget_per_ha_INR_per_season,Budget_total_INR_per_season
0,F0001,0.82,3697.01,3031.55
1,F0002,0.21,3324.54,698.15
2,F0003,1.28,3724.59,4767.47
3,F0004,1.55,3273.99,5074.69
4,F0005,0.69,3401.63,2347.13


In [3]:
# Quick descriptive statistics
print("=== 15 farmers ===")
display(farmers_15[["Farm_Size_ha", "Budget_total_INR_per_season"]].describe().round(2))

print("\n=== 25 farmers ===")
display(farmers_25[["Farm_Size_ha", "Budget_total_INR_per_season"]].describe().round(2))

=== 15 farmers ===


,Farm_Size_ha,Budget_total_INR_per_season
count,15.00,15.00
mean,0.88,3030.34
std,0.51,1835.08
min,0.21,698.15
25%,0.52,1762.66
50%,0.82,2795.11
75%,1.30,4618.73
max,1.87,6918.79



=== 25 farmers ===


,Farm_Size_ha,Budget_total_INR_per_season
count,25.00,25.00
mean,0.78,2730.72
std,0.47,1676.60
min,0.21,698.15
25%,0.43,1509.53
50%,0.65,2225.76
75%,1.03,3540.65
max,1.87,6918.79


## 2. Load practice data

In [4]:
practices = pd.read_csv(config.PRACTICES_CSV)

print(f"Number of practices : {len(practices)}")
print(f"Columns             : {list(practices.columns)}")
print()
practices

Number of practices : 20
Columns             : ['Practice', 'Net_CSP_base', 'Net_OC_per_ha', 'Base_yield_change']



,Practice,Net_CSP_base,Net_OC_per_ha,Base_yield_change
0,Zero Tillage (ZT),1.20,-3500,0.05
1,Crop Residue Retention,1.10,-3500,0.15
2,Reduced / Minimum Tillage,0.32,-2000,0.00
3,Puddling Reduction,0.25,-2000,-0.10
4,Mid-Season Drainage (MSD),0.40,-2000,-0.10
5,Alternate Wetting and Drying (AWD),0.55,-2000,0.00
6,System of Rice Intensification (SRI) Water Man...,0.60,0,0.30
7,Balanced Inorganic NPK Fertilisation,0.25,0,0.20
8,Optimised / Variable-Rate N Application,0.35,3000,0.10
9,Integrated Nutrient Management (INM) – Organic...,0.85,0,0.10


In [5]:
# Build ordered name list and name→index mapping (used by all notebooks)
practice_names = practices["Practice"].tolist()
practice_index = {name: i for i, name in enumerate(practice_names)}

M = len(practice_names)  # number of practices
print(f"M = {M} practices")
for i, p in enumerate(practice_names):
    print(f"  [{i:2d}] {p}")

M = 20 practices
  [ 0] Zero Tillage (ZT)
  [ 1] Crop Residue Retention
  [ 2] Reduced / Minimum Tillage
  [ 3] Puddling Reduction
  [ 4] Mid-Season Drainage (MSD)
  [ 5] Alternate Wetting and Drying (AWD)
  [ 6] System of Rice Intensification (SRI) Water Management
  [ 7] Balanced Inorganic NPK Fertilisation
  [ 8] Optimised / Variable-Rate N Application
  [ 9] Integrated Nutrient Management (INM) – Organic + Inorganic
  [10] Green Manure Incorporation
  [11] Farmyard Manure (FYM) Application
  [12] Compost Application
  [13] Biochar Application
  [14] Rice Straw Incorporation with N Fertiliser
  [15] Crop Rotation and Diversification
  [16] Early-Maturing / Low-Emission Variety Selection
  [17] High-Biomass / Deep-Root Variety Selection
  [18] Deep Placement of N Fertiliser
  [19] Raised Bed + Zero Tillage System


## 3. Load interaction matrices

In [6]:
def load_matrix(csv_path: str, practice_names: list, dtype=float) -> np.ndarray:
    """
    Load a 20×20 interaction matrix CSV.
    The CSV has practice names as both the index column and column headers.
    Returns a numpy array ordered according to `practice_names`.
    """
    df = pd.read_csv(csv_path, index_col=0)
    # Reindex rows and columns to match practice_names ordering
    df = df.reindex(index=practice_names, columns=practice_names)
    return df.values.astype(dtype)

Alpha = load_matrix(config.ALPHA_CSV, practice_names, dtype=float)
Beta  = load_matrix(config.BETA_CSV,  practice_names, dtype=float)
Gamma = load_matrix(config.GAMMA_CSV, practice_names, dtype=float)
Delta = load_matrix(config.DELTA_CSV, practice_names, dtype=int)

print(f"Alpha shape : {Alpha.shape}   dtype: {Alpha.dtype}")
print(f"Beta  shape : {Beta.shape}    dtype: {Beta.dtype}")
print(f"Gamma shape : {Gamma.shape}   dtype: {Gamma.dtype}")
print(f"Delta shape : {Delta.shape}   dtype: {Delta.dtype}")

Alpha shape : (20, 20)   dtype: float64
Beta  shape : (20, 20)    dtype: float64
Gamma shape : (20, 20)   dtype: float64
Delta shape : (20, 20)   dtype: int64


## 4. Validation

In [7]:
# ── 4a. NaN checks ────────────────────────────────────────────────────────────
print("NaN checks")
print("  farmers_15   :", farmers_15.isnull().sum().sum(), "NaNs")
print("  farmers_25   :", farmers_25.isnull().sum().sum(), "NaNs")
print("  practices    :", practices.isnull().sum().sum(),  "NaNs")
print("  Alpha        :", np.isnan(Alpha).sum(),           "NaNs")
print("  Beta         :", np.isnan(Beta).sum(),            "NaNs")
print("  Gamma        :", np.isnan(Gamma).sum(),           "NaNs")
print("  Delta        :", np.isnan(Delta.astype(float)).sum(), "NaNs")

NaN checks
  farmers_15   : 0 NaNs
  farmers_25   : 0 NaNs
  practices    : 0 NaNs
  Alpha        : 0 NaNs
  Beta         : 0 NaNs
  Gamma        : 0 NaNs
  Delta        : 0 NaNs


In [8]:
# ── 4b. Symmetry checks ───────────────────────────────────────────────────────
tol = config.SYMMETRY_TOL
checks = {
    "Alpha" : np.allclose(Alpha, Alpha.T, atol=tol),
    "Beta"  : np.allclose(Beta,  Beta.T,  atol=tol),
    "Gamma" : np.allclose(Gamma, Gamma.T, atol=tol),
    "Delta" : np.array_equal(Delta, Delta.T),
}
print("Symmetry checks (tolerance =", tol, ")")
all_ok = True
for name, ok in checks.items():
    status = "PASS" if ok else "FAIL"
    print(f"  {name:6s} : {status}")
    if not ok:
        all_ok = False

assert all_ok, "One or more matrices failed the symmetry check — inspect the CSVs."

Symmetry checks (tolerance = 1e-08 )
  Alpha  : PASS
  Beta   : PASS
  Gamma  : PASS
  Delta  : PASS


In [9]:
# ── 4c. Zero-diagonal checks ──────────────────────────────────────────────────
print("Zero-diagonal checks")
for name, mat in [("Alpha", Alpha), ("Beta", Beta), ("Gamma", Gamma), ("Delta", Delta)]:
    diag_sum = np.abs(np.diag(mat)).sum()
    status = "PASS" if diag_sum == 0 else f"FAIL (sum={diag_sum})"
    print(f"  {name:6s} diagonal : {status}")

Zero-diagonal checks
  Alpha  diagonal : PASS
  Beta   diagonal : PASS
  Gamma  diagonal : PASS
  Delta  diagonal : PASS


In [10]:
# ── 4d. Delta value check — must be binary (0 or 1) ──────────────────────────
unique_delta = np.unique(Delta)
assert set(unique_delta).issubset({0, 1}), f"Delta contains non-binary values: {unique_delta}"
print(f"Delta values : {unique_delta}  — binary check PASS")

Delta values : [0 1]  — binary check PASS


In [11]:
# ── 4e. Budget sanity — every budget must be > 0 ──────────────────────────────
for label, df in [("15 farmers", farmers_15), ("25 farmers", farmers_25)]:
    neg = (df["Budget_total_INR_per_season"] <= 0).sum()
    assert neg == 0, f"{label}: {neg} farmers have non-positive budget"
    print(f"Budget check ({label}): all positive — PASS")

Budget check (15 farmers): all positive — PASS
Budget check (25 farmers): all positive — PASS


## 5. Incompatible practice pairs

In [12]:
# Collect upper-triangle pairs where Delta_jk = 1
incompatible_pairs = []
for j in range(M):
    for k in range(j + 1, M):
        if Delta[j, k] == 1:
            incompatible_pairs.append((j, k, practice_names[j], practice_names[k]))

print(f"Total incompatible pairs: {len(incompatible_pairs)}")
print()
print("-" * 80)
for idx_j, idx_k, p1, p2 in incompatible_pairs:
    print(f"  [{idx_j:2d}] {p1}")
    print(f"  [{idx_k:2d}] {p2}")
    print(f"  → These two practices cannot be co-adopted (Δ = 1)")
    print("-" * 80)

Total incompatible pairs: 2

--------------------------------------------------------------------------------
  [ 0] Zero Tillage (ZT)
  [19] Raised Bed + Zero Tillage System
  → These two practices cannot be co-adopted (Δ = 1)
--------------------------------------------------------------------------------
  [ 7] Balanced Inorganic NPK Fertilisation
  [ 9] Integrated Nutrient Management (INM) – Organic + Inorganic
  → These two practices cannot be co-adopted (Δ = 1)
--------------------------------------------------------------------------------


## 6. Summary statistics on interaction matrices

In [13]:
def upper_triangle_stats(mat: np.ndarray, name: str, unit: str) -> None:
    """
    Print statistics on the upper triangle (j < k) of a symmetric matrix.
    Incompatible pairs (Delta_jk=1) are excluded because their interaction
    values are structurally zero and irrelevant to the optimiser.
    """
    vals = []
    for j in range(M):
        for k in range(j + 1, M):
            if Delta[j, k] == 0:   # skip incompatible pairs
                vals.append(mat[j, k])
    vals = np.array(vals)
    nonzero = (vals != 0).sum()
    print(f"{name} ({unit}) — compatible pairs only")
    print(f"  n pairs  : {len(vals)}")
    print(f"  nonzero  : {nonzero}")
    print(f"  min      : {vals.min():.4f}")
    print(f"  max      : {vals.max():.4f}")
    print(f"  mean     : {vals.mean():.4f}")
    print(f"  positive : {(vals > 0).sum()}   negative: {(vals < 0).sum()}")
    print()

upper_triangle_stats(Alpha, "Alpha", "tCO2e/ha/season")
upper_triangle_stats(Beta,  "Beta",  "INR/ha/season")
upper_triangle_stats(Gamma, "Gamma", "tons/ha/season")

Alpha (tCO2e/ha/season) — compatible pairs only
  n pairs  : 188
  nonzero  : 188
  min      : -0.0993
  max      : 0.2291
  mean     : 0.0147
  positive : 102   negative: 86

Beta (INR/ha/season) — compatible pairs only
  n pairs  : 188
  nonzero  : 188
  min      : -334.5645
  max      : 205.1699
  mean     : -19.3133
  positive : 70   negative: 118

Gamma (tons/ha/season) — compatible pairs only
  n pairs  : 188
  nonzero  : 188
  min      : -0.0757
  max      : 0.1362
  mean     : 0.0098
  positive : 114   negative: 74



## 7. Config parameter echo

In [14]:
print("Economic parameters loaded from config.py")
print("-" * 50)
print(f"  CCP (carbon credit price)   : {config.CCP:>10,} INR/tCO2e")
print(f"  PADDY_PRICE                 : {config.PADDY_PRICE:>10,} INR/ton")
print(f"  FIXED_MRV                   : {config.FIXED_MRV:>10,} INR")
print(f"  VARIABLE_MRV                : {config.VARIABLE_MRV:>10,} INR/ha^delta")
print(f"  DELTA_MRV                   : {config.DELTA_MRV:>10}")
print(f"  FIXED_T                     : {config.FIXED_T:>10,} INR")
print(f"  VARIABLE_T                  : {config.VARIABLE_T:>10,} INR/farmer")
print("-" * 50)

# Quick sanity: what does solo certification cost for a 1-ha farmer?
solo_mrv = config.mrv_cost(1.0)
solo_t   = config.transaction_cost(1)
print(f"\nSolo certification cost (1 ha farmer):")
print(f"  MRV cost   : {solo_mrv:,.0f} INR")
print(f"  Trans cost : {solo_t:,.0f} INR")
print(f"  Total      : {solo_mrv + solo_t:,.0f} INR")

# And for a 10-ha coalition of 5 farmers?
coal_mrv = config.mrv_cost(10.0)
coal_t   = config.transaction_cost(5)
print(f"\nCoalition certification cost (5 farmers, 10 ha total):")
print(f"  MRV cost   : {coal_mrv:,.0f} INR")
print(f"  Trans cost : {coal_t:,.0f} INR")
print(f"  Total      : {coal_mrv + coal_t:,.0f} INR")

Economic parameters loaded from config.py
--------------------------------------------------
  CCP (carbon credit price)   :      1,500 INR/tCO2e
  PADDY_PRICE                 :     22,000 INR/ton
  FIXED_MRV                   :      5,000 INR
  VARIABLE_MRV                :      2,000 INR/ha^delta
  DELTA_MRV                   :        0.7
  FIXED_T                     :      2,000 INR
  VARIABLE_T                  :        500 INR/farmer
--------------------------------------------------

Solo certification cost (1 ha farmer):
  MRV cost   : 7,000 INR
  Trans cost : 2,500 INR
  Total      : 9,500 INR

Coalition certification cost (5 farmers, 10 ha total):
  MRV cost   : 15,024 INR
  Trans cost : 4,500 INR
  Total      : 19,524 INR


## 8. Save processed inputs pickle

In [15]:
# Build the config snapshot dict (only serialisable values — no functions)
config_snapshot = {
    "CCP"          : config.CCP,
    "PADDY_PRICE"  : config.PADDY_PRICE,
    "FIXED_MRV"    : config.FIXED_MRV,
    "VARIABLE_MRV" : config.VARIABLE_MRV,
    "DELTA_MRV"    : config.DELTA_MRV,
    "FIXED_T"      : config.FIXED_T,
    "VARIABLE_T"   : config.VARIABLE_T,
    "SYMMETRY_TOL" : config.SYMMETRY_TOL,
    "EPSILON_MAX"  : config.EPSILON_MAX,
}

optimization_inputs = {
    # Farmer DataFrames
    "farmers_15"      : farmers_15,
    "farmers_25"      : farmers_25,

    # Practice DataFrame
    "practices"       : practices,

    # Practice naming / indexing
    "practice_names"  : practice_names,   # list[str], length M
    "practice_index"  : practice_index,   # dict str→int

    # Interaction matrices (all M×M numpy arrays)
    "Alpha"           : Alpha,   # tCO2e/ha/season — sequestration synergies
    "Beta"            : Beta,    # INR/ha/season   — cost interactions
    "Gamma"           : Gamma,   # tons/ha/season  — yield interactions
    "Delta"           : Delta,   # binary          — incompatibility (0/1)

    # Incompatible pair list for fast lookup [(j, k, name_j, name_k), ...]
    "incompatible_pairs" : incompatible_pairs,

    # Economic parameters
    "config"          : config_snapshot,
}

out_path = Path(config.INPUTS_PKL)
with open(out_path, "wb") as f:
    pickle.dump(optimization_inputs, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved: {out_path.resolve()}")
print(f"Keys  : {list(optimization_inputs.keys())}")

Saved: /Users/rohitsuresh/Desktop/CGT_Carbon_farming/data/processed/Experiment_01_Baseline/optimization_inputs.pkl
Keys  : ['farmers_15', 'farmers_25', 'practices', 'practice_names', 'practice_index', 'Alpha', 'Beta', 'Gamma', 'Delta', 'incompatible_pairs', 'config']


## 9. Reload and verify the pickle

In [16]:
with open(config.INPUTS_PKL, "rb") as f:
    loaded = pickle.load(f)

# Structural checks on the reloaded object
assert set(loaded.keys()) == set(optimization_inputs.keys()), "Key mismatch after reload"
assert np.array_equal(loaded["Alpha"], Alpha), "Alpha mismatch after reload"
assert np.array_equal(loaded["Delta"], Delta), "Delta mismatch after reload"
assert len(loaded["practice_names"]) == M,     "practice_names length mismatch"
assert len(loaded["farmers_15"]) == 15,         "farmers_15 row count mismatch"
assert len(loaded["farmers_25"]) == 25,         "farmers_25 row count mismatch"

print("Pickle reload checks: all PASS")
print(f"\nFile size: {out_path.stat().st_size / 1024:.1f} KB")

Pickle reload checks: all PASS

File size: 16.9 KB


## 10. Final summary

In [17]:
print("=" * 60)
print("NOTEBOOK 01 COMPLETE")
print("=" * 60)
print(f"  Practices loaded         : {M}")
print(f"  Incompatible pairs       : {len(incompatible_pairs)}")
print(f"  Farmers (15-set)         : {len(farmers_15)}")
print(f"  Farmers (25-set)         : {len(farmers_25)}")
print(f"  Matrices validated       : Alpha, Beta, Gamma, Delta")
print(f"  Output pickle            : {config.INPUTS_PKL}")
print()


NOTEBOOK 01 COMPLETE
  Practices loaded         : 20
  Incompatible pairs       : 2
  Farmers (15-set)         : 15
  Farmers (25-set)         : 25
  Matrices validated       : Alpha, Beta, Gamma, Delta
  Output pickle            : /Users/rohitsuresh/Desktop/CGT_Carbon_farming/data/processed/Experiment_01_Baseline/optimization_inputs.pkl

